In [2]:
import json
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

OUTPUT_PATH = "../data/processed/"
BIG5_LABELS = ["agreeableness_label", "openness_label", 
                "conscientiousness_label", "extraversion_label", 
                "neuroticism_label"]

BERT_MODEL   = "bert-base-uncased"
MAX_TOKENS   = 128   # per comment
MAX_COMMENTS = 50    # per user

In [3]:
profiles  = pd.read_csv(OUTPUT_PATH + "profiles_labeled.csv")
comments  = pd.read_csv(OUTPUT_PATH + "comments_capped.csv")

with open(OUTPUT_PATH + "splits.json") as f:
    splits = json.load(f)

with open(OUTPUT_PATH + "pos_weights.json") as f:
    pw = json.load(f)
    pos_weights = torch.tensor(pw["pos_weights"], dtype=torch.float)

print(f"Profiles: {len(profiles)}")
print(f"Comments: {len(comments)}")
print(f"pos_weights: {pos_weights}")

Profiles: 1568
Comments: 67450
pos_weights: tensor([1.2893, 0.4018, 1.6313, 1.8438, 0.9711])


In [4]:
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)

class PANDORADataset(Dataset):
    def __init__(self, author_list, profiles_df, comments_df,
                 max_comments=MAX_COMMENTS, max_tokens=MAX_TOKENS):
        
        self.max_comments = max_comments
        self.max_tokens   = max_tokens
        
        # Filter to only authors in this split
        self.profiles = profiles_df[
            profiles_df["author"].isin(author_list)
        ].reset_index(drop=True)
        
        # Group comments by author once — much faster than filtering per __getitem__
        self.comments_by_author = (
            comments_df[comments_df["author"].isin(author_list)]
            .groupby("author")["body"]
            .apply(list)
            .to_dict()
        )
    
    def __len__(self):
        return len(self.profiles)
    
    def __getitem__(self, idx):
        row    = self.profiles.iloc[idx]
        author = row["author"]
        
        # Labels: shape (5,)
        labels = torch.tensor(
            row[BIG5_LABELS].values.astype(float),
            dtype=torch.float
        )
        
        # Get this user's comments (already capped at 50 in pipeline)
        user_comments = self.comments_by_author.get(author, [""])
        
        # Tokenize each comment independently
        # Output shape per comment: (max_tokens,)
        # We'll stack them into (max_comments, max_tokens)
        input_ids_list      = []
        attention_mask_list = []
        
        for comment in user_comments[:self.max_comments]:
            encoded = tokenizer(
                str(comment),
                max_length=self.max_tokens,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )
            input_ids_list.append(encoded["input_ids"].squeeze(0))        # (max_tokens,)
            attention_mask_list.append(encoded["attention_mask"].squeeze(0))
        
        # Pad user to max_comments if they have fewer comments
        # Padding comment = all zeros (BERT will ignore via attention mask)
        num_comments = len(input_ids_list)
        
        while len(input_ids_list) < self.max_comments:
            input_ids_list.append(torch.zeros(self.max_tokens, dtype=torch.long))
            attention_mask_list.append(torch.zeros(self.max_tokens, dtype=torch.long))
        
        input_ids      = torch.stack(input_ids_list)       # (max_comments, max_tokens)
        attention_masks = torch.stack(attention_mask_list) # (max_comments, max_tokens)
        
        # comment_mask: which comment slots are real vs padding
        # Shape: (max_comments,) — True = real comment, False = padding
        comment_mask = torch.zeros(self.max_comments, dtype=torch.bool)
        comment_mask[:num_comments] = True
        
        return {
            "input_ids":       input_ids,        # (50, 128)
            "attention_masks": attention_masks,   # (50, 128)
            "comment_mask":    comment_mask,      # (50,)
            "labels":          labels             # (5,)
        }

In [5]:
train_dataset = PANDORADataset(splits["train"], profiles, comments)
val_dataset   = PANDORADataset(splits["val"],   profiles, comments)
test_dataset  = PANDORADataset(splits["test"],  profiles, comments)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# Inspect one sample
sample = train_dataset[0]
print(f"\ninput_ids shape:       {sample['input_ids'].shape}")
print(f"attention_masks shape: {sample['attention_masks'].shape}")
print(f"comment_mask shape:    {sample['comment_mask'].shape}")
print(f"labels shape:          {sample['labels'].shape}")
print(f"labels:                {sample['labels']}")
print(f"real comments in sample: {sample['comment_mask'].sum().item()}")


Train: 1092 | Val: 243 | Test: 233

input_ids shape:       torch.Size([50, 128])
attention_masks shape: torch.Size([50, 128])
comment_mask shape:    torch.Size([50])
labels shape:          torch.Size([5])
labels:                tensor([0., 1., 1., 1., 0.])
real comments in sample: 5


In [6]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=8, shuffle=False, num_workers=0)

# Verify one batch
batch = next(iter(train_loader))
print("One batch shapes:")
print(f"  input_ids:       {batch['input_ids'].shape}")       # (8, 50, 128)
print(f"  attention_masks: {batch['attention_masks'].shape}") # (8, 50, 128)
print(f"  comment_mask:    {batch['comment_mask'].shape}")    # (8, 50)
print(f"  labels:          {batch['labels'].shape}")          # (8, 5)

One batch shapes:
  input_ids:       torch.Size([8, 50, 128])
  attention_masks: torch.Size([8, 50, 128])
  comment_mask:    torch.Size([8, 50])
  labels:          torch.Size([8, 5])


In [8]:
import torch
import torch.nn as nn
from transformers import AutoModel

class LabelAttention(nn.Module):
    """
    Given comment embeddings and 5 learnable label vectors,
    compute a weighted sum of comments for each label.
    
    Input:
        comment_embs  : (batch, num_comments, hidden)
        comment_mask  : (batch, num_comments) — True = real, False = padding
    Output:
        context       : (batch, num_labels, hidden)
    """
    def __init__(self, hidden_size, num_labels=5):
        super().__init__()
        # One learnable embedding per personality trait
        self.label_embeddings = nn.Embedding(num_labels, hidden_size)
        self.num_labels = num_labels
    
    def forward(self, comment_embs, comment_mask):
        # label_vecs: (num_labels, hidden)
        label_idx  = torch.arange(self.num_labels, device=comment_embs.device)
        label_vecs = self.label_embeddings(label_idx)
        
        # scores: (batch, num_labels, num_comments)
        # Each label vector dot-products with every comment embedding
        scores = torch.matmul(label_vecs, comment_embs.transpose(1, 2))
        
        # Mask padding comments before softmax
        # comment_mask: (batch, num_comments) → (batch, 1, num_comments)
        mask = comment_mask.unsqueeze(1)  
        scores = scores.masked_fill(~mask, float("-inf"))
        
        # Softmax over comment dimension → attention weights
        attn_weights = torch.softmax(scores, dim=-1)  # (batch, num_labels, num_comments)
        
        # Weighted sum of comment embeddings per label
        # context: (batch, num_labels, hidden)
        context = torch.matmul(attn_weights, comment_embs)
        
        return context, attn_weights


class HTLA(nn.Module):
    """
    Hierarchical Transformer with Label Attention for Big Five prediction.
    
    Flow:
        input_ids, attention_masks : (batch, num_comments, seq_len)
        → BERT per comment         : (batch, num_comments, hidden)
        → Label Attention          : (batch, num_labels, hidden)
        → Linear per label         : (batch, num_labels) = logits
    """
    def __init__(self, bert_model_name=BERT_MODEL, num_labels=5, dropout=0.1):
        super().__init__()
        self.bert       = AutoModel.from_pretrained(bert_model_name)
        hidden_size     = self.bert.config.hidden_size  # 768 for bert-base
        
        self.dropout    = nn.Dropout(dropout)
        self.label_attn = LabelAttention(hidden_size, num_labels)
        
        # One classifier head per trait
        self.classifiers = nn.ModuleList([
            nn.Linear(hidden_size, 1) for _ in range(num_labels)
        ])
    
    def forward(self, input_ids, attention_masks, comment_mask):
        batch_size, num_comments, seq_len = input_ids.shape
        
        # --- Step 1: Encode each comment with BERT ---
        # Reshape to (batch * num_comments, seq_len) so BERT sees flat sequences
        input_ids_flat      = input_ids.view(batch_size * num_comments, seq_len)
        attention_masks_flat = attention_masks.view(batch_size * num_comments, seq_len)
        
        bert_out = self.bert(
            input_ids=input_ids_flat,
            attention_mask=attention_masks_flat
        )
        
        # Take [CLS] token as comment representation: (batch*num_comments, hidden)
        cls_embeddings = bert_out.last_hidden_state[:, 0, :]
        cls_embeddings = self.dropout(cls_embeddings)
        
        # Reshape back: (batch, num_comments, hidden)
        comment_embs = cls_embeddings.view(batch_size, num_comments, -1)
        
        # --- Step 2: Label Attention ---
        # context: (batch, num_labels, hidden)
        context, attn_weights = self.label_attn(comment_embs, comment_mask)
        context = self.dropout(context)
        
        # --- Step 3: Classify each trait ---
        # logits: (batch, num_labels)
        logits = torch.cat([
            self.classifiers[i](context[:, i, :])  # (batch, 1)
            for i in range(len(self.classifiers))
        ], dim=1)
        
        return logits, attn_weights

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = HTLA(bert_model_name=BERT_MODEL, num_labels=5, dropout=0.1)
model = model.to(device)

# Count parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Forward pass on one batch
batch = next(iter(train_loader))
input_ids      = batch["input_ids"].to(device)
attention_masks = batch["attention_masks"].to(device)
comment_mask   = batch["comment_mask"].to(device)
labels         = batch["labels"].to(device)

with torch.no_grad():
    logits, attn_weights = model(input_ids, attention_masks, comment_mask)

print(f"\nlogits shape:       {logits.shape}")        # (8, 5)
print(f"attn_weights shape: {attn_weights.shape}")    # (8, 5, 50)
print(f"labels shape:       {labels.shape}")          # (8, 5)
print(f"\nSample logits:\n{logits[0]}")

Using device: cpu


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

c:\Users\Tamara\AppData\Local\pypoetry\Cache\virtualenvs\tugasakhir-pfEGgxcN-py3.12\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tamara\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total parameters:     109,489,925
Trainable parameters: 109,489,925

logits shape:       torch.Size([8, 5])
attn_weights shape: torch.Size([8, 5, 50])
labels shape:       torch.Size([8, 5])

Sample logits:
tensor([-0.4128, -0.2718, -0.4987,  0.0074,  0.0619])


In [10]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights.to(device))

# Test loss on the batch
loss = criterion(logits, labels)
print(f"Loss on first batch (untrained model): {loss.item():.4f}")

Loss on first batch (untrained model): 0.7464
